# RL-Driven Quantum Pulse Optimization
### An Educational Tutorial for Superconducting Qubit Control

---

**Author:** Quantum Control Engineering Portfolio Project  
**Stack:** QuTiP · Gymnasium · Stable-Baselines3 (PPO) · NumPy · Matplotlib

---

## What We Are Building

We will train a **Reinforcement Learning agent** to find the optimal microwave pulse shape that drives a superconducting qubit from the **ground state |0⟩ to the excited state |1⟩** (an X-gate) in minimum time, maximizing fidelity under realistic quantum noise.

This is not just an academic exercise — this approach is directly relevant to how **IQM, Alice & Bob, IBM, and Google** calibrate quantum gates on real hardware.

### Roadmap
1. Physics of a superconducting qubit (two-level system)
2. Open quantum systems: the Lindblad master equation
3. Why RL? Classical vs. RL approaches to pulse optimization
4. The Markov Decision Process (MDP) formulation
5. Environment walkthrough (`quantum_env.py`)
6. Training a PPO agent (short demo loop)
7. Visualizing results: Bloch sphere + pulse shape

---
## 1. Physics: The Superconducting Qubit as a Two-Level System

A **transmon** superconducting qubit behaves, to a very good approximation, as a two-level quantum system (qubit) described by a 2×2 complex density matrix ρ.

### The System Hamiltonian

In the **rotating-wave approximation (RWA)** — the standard frame used in microwave engineering of qubits — the total Hamiltonian is:

$$H(t) = \underbrace{\frac{\omega_q}{2}\sigma_z}_{H_0 \text{ (drift)}} + \underbrace{\frac{\Omega(t)}{2}\sigma_x}_{H_c \text{ (control)}}$$

| Symbol | Meaning |
|--------|---------|
| $\omega_q$ | Qubit transition frequency (set to 0 at resonance in rotating frame) |
| $\sigma_z, \sigma_x$ | Pauli matrices |
| $\Omega(t)$ | Time-varying microwave drive amplitude — **this is what the RL agent controls** |

The **Bloch sphere** is a geometric representation where any pure qubit state maps to a point on the unit sphere:
- North pole: |1⟩ (excited)
- South pole: |0⟩ (ground)
- Equator: superpositions

An **X-gate** rotates the Bloch vector by π around the X-axis: |0⟩ → |1⟩.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

# Print the Pauli matrices to build intuition
print('sigma_x (control coupling):')
print(qt.sigmax())
print('\nsigma_z (drift):')
print(qt.sigmaz())
print('\n|0> ground state:')
print(qt.basis(2, 0))
print('\n|1> excited state (target):')
print(qt.basis(2, 1))

---
## 2. Open Quantum Systems: The Lindblad Master Equation

A real qubit is never perfectly isolated. It couples to its electromagnetic environment, causing **decoherence** — the gradual loss of quantum information. We model this with the **Lindblad master equation**:

$$\frac{d\rho}{dt} = -i[H(t), \rho] + \sum_k \left(L_k \rho L_k^\dagger - \frac{1}{2}\{L_k^\dagger L_k, \rho\}\right)$$

The first term is unitary (coherent) evolution under H(t). The sum captures **dissipation** via collapse operators $L_k$:

### Decoherence Channels

**T1 — Energy relaxation** (amplitude damping):
$$L_1 = \sqrt{\Gamma_1}\, \sigma_- \quad \text{where } \Gamma_1 = 1/T_1$$
The qubit spontaneously emits a photon and falls from |1⟩ to |0⟩. Typical T1 ≈ 1–100 µs for transmons.

**T2 — Dephasing** (phase randomization):
$$L_2 = \sqrt{\Gamma_\phi}\, \frac{\sigma_z}{2} \quad \text{where } \Gamma_\phi = \frac{1}{T_2} - \frac{\Gamma_1}{2}$$
Random fluctuations in the qubit frequency cause phase drift. Typical T2 ≈ 0.5–10 µs.

### Why does this matter for pulse optimization?

The **longer the gate takes**, the more decoherence accumulates. This creates a fundamental tension:
- **Too slow** → fidelity lost to T1/T2 decay
- **Too fast** → bandwidth limits of the control electronics, leakage to higher levels

RL naturally discovers pulses that balance this trade-off.

In [ ]:
# Simulate free decay of an excited qubit (no drive) to visualize T1
import qutip as qt
import numpy as np
import matplotlib.pyplot as plt

T1 = 1000.0   # ns
T2 = 500.0    # ns
Gamma1 = 1.0 / T1
Gamma_phi = 1.0/T2 - Gamma1/2

c_ops = [
    np.sqrt(Gamma1) * qt.destroy(2),         # T1: sigma_minus
    np.sqrt(Gamma_phi) * qt.sigmaz() / 2,    # T2: dephasing
]

# Start in |1> excited state, no drive (H=0)
psi0 = qt.ket2dm(qt.basis(2, 1))
tlist = np.linspace(0, 3000, 300)  # 0 to 3 µs

result = qt.mesolve(qt.Qobj([[0,0],[0,0]]), psi0, tlist, c_ops,
                    [qt.sigmaz()])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tlist, result.expect[0], color='#E53935', linewidth=2, label='⟨Z⟩ (population)')
ax.axhline(-1, color='gray', linestyle='--', alpha=0.5, label='|0⟩ ground')
ax.axhline(1,  color='gray', linestyle='--', alpha=0.5, label='|1⟩ excited')
ax.set_xlabel('Time (ns)', fontsize=12)
ax.set_ylabel('⟨σ_z⟩', fontsize=12)
ax.set_title(f'T1 Decay of Excited Qubit  (T1={T1:.0f} ns, T2={T2:.0f} ns)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'At t=T1={T1:.0f} ns, <Z> ≈ {result.expect[0][100]:.3f} (expected ≈ -0.26 for mixed decay)')

---
## 3. Why RL? Classical vs. RL Pulse Optimization

### Classical Approach: GRAPE (Gradient Ascent Pulse Engineering)

GRAPE propagates the system forward in time and backpropagates the fidelity gradient through the pulse sequence. It is powerful but has limitations:

- Requires a **differentiable simulator** — hard to use directly on hardware
- Sensitive to **initial guess** — can get stuck in local optima
- Doesn't adapt online to **time-varying qubit parameters** (drift)

### RL Approach: Policy Gradient (PPO)

The RL agent treats the qubit as a **black box** environment and learns by trial and error:

```
  Agent ──Ω(t)──▶ Qubit ──[Bloch state]──▶ Agent
       ◀──reward (fidelity)──────────────────┘
```

| Property | GRAPE | RL (PPO) |
|----------|-------|----------|
| Gradient required? | Yes | No |
| Works on hardware? | Hard | Yes (model-free) |
| Handles stochastic noise? | Poorly | Naturally |
| Online adaptation? | No | Yes (fine-tune policy) |
| Sample efficiency | High | Medium (but improving) |

### Why PPO specifically?

PPO is a **clipped surrogate** policy gradient method that prevents destructively large policy updates:

$$L^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t \right) \right]$$

This stability makes it ideal for quantum control where reward signals are sparse (only at the end of each episode).

---
## 4. The Markov Decision Process (MDP) Formulation

To apply RL, we cast pulse optimization as an MDP with the following components:

### State Space  $s_t \in \mathbb{R}^3$

The state is the **Bloch vector** — the three expectation values of the Pauli operators:
$$s_t = [\langle\sigma_x\rangle,\ \langle\sigma_y\rangle,\ \langle\sigma_z\rangle]$$

This is a complete description of the qubit's mixed state (given we model the environment as Markovian). Ranges in $[-1, 1]^3$, with $|s|\leq 1$.

### Action Space  $a_t \in [-1, 1]$

The agent outputs a **normalized drive amplitude** at each timestep, scaled to physical units:
$$\Omega(t) = a_t \cdot \Omega_{\max}$$

This is a **continuous action** — PPO handles this with a Gaussian policy.

### Reward Function

$$r_t = \begin{cases} -\lambda \cdot \Omega(t)^2 & t < T \text{ (energy penalty)} \\ F(\rho_T, |1\rangle) - \lambda \cdot \Omega(T)^2 & t = T \text{ (fidelity bonus)} \end{cases}$$

- $\lambda = 0.01$ balances energy efficiency vs. fidelity
- Fidelity $F = \langle 1|\rho_T|1\rangle \in [0, 1]$ is the probability of measuring |1⟩ at the end

### Episode Structure

Each episode = one X-gate attempt:
- **Start**: qubit in $|0\rangle$ (Bloch: $[0, 0, -1]$)
- **Steps**: N=50 steps × dt=1 ns = **50 ns gate**
- **End**: measure fidelity vs. $|1\rangle$ (Bloch: $[0, 0, +1]$)

In [ ]:
# Verify our environment setup
import sys
sys.path.insert(0, '.')  # ensure local modules are importable

from quantum_env import QuantumPulseEnv
from utils import QUBIT_PARAMS

env = QuantumPulseEnv(n_steps=50, dt=1.0, omega_max=0.1, energy_penalty=0.01)

print('Environment summary')
print('=' * 40)
print(f'  Observation space : {env.observation_space}')
print(f'  Action space      : {env.action_space}')
print(f'  Gate time         : {env.n_steps * env.dt:.0f} ns')
print(f'  Omega_max         : {env.omega_max:.3f} x 2pi MHz')
print(f'  T1                : {QUBIT_PARAMS["T1"]:.0f} ns')
print(f'  T2                : {QUBIT_PARAMS["T2"]:.0f} ns')

# Run one random episode to verify it works
obs, _ = env.reset()
print(f'\nInitial Bloch vector (|0>): {obs}')
print('  Expected: [0, 0, -1] (south pole)')

total_reward = 0
for _ in range(env.n_steps):
    action = env.action_space.sample()
    obs, reward, done, _, info = env.step(action)
    total_reward += reward

print(f'\nFinal Bloch vector (random policy): {obs}')
print(f'Final fidelity (random policy)    : {info["fidelity"]:.4f}')
print(f'Total reward                       : {total_reward:.4f}')

---
## 5. Environment Deep Dive

### How `quantum_env.py` works step by step

```python
def step(self, action):
    # 1. Decode action → physical amplitude
    amplitude = action[0] * self.omega_max

    # 2. Build time-dependent Hamiltonian
    H = [H0, [Hc, lambda t, args: amplitude]]

    # 3. Integrate Lindblad equation for dt nanoseconds
    result = qt.mesolve(H, rho, tlist=[0, dt], c_ops=c_ops)
    rho_new = result.states[-1]

    # 4. Compute observation (Bloch vector)
    obs = [expect(sx, rho_new), expect(sy, rho_new), expect(sz, rho_new)]

    # 5. Compute reward
    reward = -energy_penalty * amplitude**2
    if done:
        reward += fidelity(rho_new, |1>)

    return obs, reward, done, info
```

### The Markovian assumption

The Lindblad equation assumes **Markovian (memoryless) noise** — the environment's correlation time is much shorter than the gate time. This is an excellent approximation for transmon qubits at millikelvin temperatures where:
- Phonon bath correlation time ~ ps
- Gate time ~ 10–100 ns

This also means the **Bloch vector is a sufficient state** for the MDP — no memory of past states is needed.

In [ ]:
# Visualize the Hamiltonians
from utils import build_hamiltonians, build_collapse_operators, QUBIT_PARAMS
import qutip as qt

H0, Hc = build_hamiltonians(QUBIT_PARAMS)
c_ops = build_collapse_operators(QUBIT_PARAMS)

print('H0 (drift Hamiltonian) — omega_q/2 * sigma_z:')
print(H0)
print('\nHc (control Hamiltonian) — 1/2 * sigma_x:')
print(Hc)
print(f'\nNumber of collapse operators: {len(c_ops)}')
print('c_ops[0] — T1 amplitude damping (sigma_minus):')
print(c_ops[0])
print('c_ops[1] — T2 dephasing (sigma_z/2):')
print(c_ops[1])

---
## 6. Training a PPO Agent (Short Demo)

The full training run (200 k steps) takes ~15–30 min. Here we run a **short demo** (5 000 steps) just to verify the pipeline. Use `train.py` for full training.

### PPO Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `n_steps` | 200 | Rollout buffer length |
| `batch_size` | 64 | Mini-batch for updates |
| `n_epochs` | 10 | Gradient steps per rollout |
| `clip_range` | 0.2 | PPO clip ε |
| `ent_coef` | 0.01 | Entropy bonus for exploration |
| `learning_rate` | 3e-4 | Adam optimizer |
| Network | MLP [64, 64] | Sufficient for 3-D state space |

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from quantum_env import QuantumPulseEnv

# Create environment
env = Monitor(QuantumPulseEnv(n_steps=50, dt=1.0, omega_max=0.1, energy_penalty=0.01))

# Build PPO agent
model = PPO(
    policy='MlpPolicy',
    env=env,
    learning_rate=3e-4,
    n_steps=200,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=dict(net_arch=[64, 64]),
    seed=42,
    verbose=0,
)

print('PPO agent created!')
print(f'Policy architecture: {model.policy}')
print(f'Total parameters: {sum(p.numel() for p in model.policy.parameters()):,}')

In [ ]:
# Short training demo — 5000 timesteps (~100 episodes)
# NOTE: For real results, run: python train.py --timesteps 200000

from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

class FidelityTracker(BaseCallback):
    def __init__(self):
        super().__init__()
        self.fidelities = []
        self.episode_rewards = []
        self._ep_reward = 0.0

    def _on_step(self):
        self._ep_reward += float(self.locals['rewards'][0])
        if self.locals['dones'][0]:
            info = self.locals['infos'][0]
            if 'fidelity' in info:
                self.fidelities.append(info['fidelity'])
            self.episode_rewards.append(self._ep_reward)
            self._ep_reward = 0.0
        return True

tracker = FidelityTracker()

print('Training for 5000 timesteps (demo)...')
model.learn(total_timesteps=5000, callback=tracker, progress_bar=True)
print('Demo training complete!')

if tracker.fidelities:
    print(f'\nFidelity over training:')
    print(f'  First 10 episodes mean : {np.mean(tracker.fidelities[:10]):.4f}')
    print(f'  Last  10 episodes mean : {np.mean(tracker.fidelities[-10:]):.4f}')
    print(f'  (Run python train.py for full 200k timestep training)')

In [ ]:
# Plot training progress
import matplotlib.pyplot as plt
import numpy as np

if tracker.fidelities:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(tracker.fidelities, alpha=0.6, color='#42A5F5')
    if len(tracker.fidelities) >= 10:
        rolling = np.convolve(tracker.fidelities, np.ones(10)/10, mode='valid')
        ax1.plot(np.arange(9, len(tracker.fidelities)), rolling,
                 color='#1565C0', linewidth=2, label='10-ep rolling avg')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Gate Fidelity')
    ax1.set_title('Fidelity During Training (demo)')
    ax1.set_ylim([0, 1])
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(tracker.episode_rewards, alpha=0.6, color='#66BB6A')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Total Episode Reward')
    ax2.set_title('Episode Rewards (demo)')
    ax2.grid(True, alpha=0.3)

    plt.suptitle('PPO Training Progress (5000 steps demo)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 7. Evaluating the Agent: Pulse Shape + Bloch Sphere

After training (or using the demo model), we can:
1. Run one deterministic episode with the trained policy
2. Record the pulse amplitudes Ω(t)
3. Re-simulate the trajectory and plot on the Bloch sphere

In [ ]:
# Evaluate the demo-trained agent
from quantum_env import QuantumPulseEnv
from utils import plot_pulse_and_bloch
import numpy as np

eval_env = QuantumPulseEnv(n_steps=50, dt=1.0, omega_max=0.1, energy_penalty=0.01)

obs, _ = eval_env.reset()
done = False
info = {}

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    done = terminated or truncated

fidelity = info.get('fidelity', 0.0)
amplitudes = np.array(info.get('pulse_amplitudes', []))

print(f'Demo agent fidelity: {fidelity:.4f}')
print(f'  (Note: demo only trained for 5000 steps — expect ~0.5–0.8)')
print(f'  (Full training at 200k steps targets >0.97 fidelity)')

# Simulate Bloch trajectory
trajectory = eval_env.get_bloch_trajectory(amplitudes.tolist())
print(f'Trajectory shape: {trajectory.shape}  (n_steps+1, 3)')
print(f'Start point (|0>): {trajectory[0]}')
print(f'End   point      : {trajectory[-1]}')

In [ ]:
# Plot pulse shape + Bloch sphere trajectory
from utils import plot_pulse_and_bloch

plot_pulse_and_bloch(
    amplitudes=amplitudes,
    trajectory=trajectory,
    dt=1.0,
    fidelity=fidelity,
)

---
## 8. Next Steps & Extensions

This project demonstrates the core pipeline. Here are natural extensions that would be relevant for quantum hardware internships:

### Physics Extensions
- **Detuning robustness**: Add qubit frequency uncertainty $\delta\omega_q$ sampled each episode — the agent learns a pulse robust to frequency drift.
- **DRAG correction**: Add a Y-quadrature action to suppress leakage to the |2⟩ state (full transmon).
- **Two-qubit gates**: Extend to a 4-level system (2 qubits) and optimize a CNOT gate.

### ML Extensions
- **SAC (Soft Actor-Critic)**: Better sample efficiency than PPO for continuous actions.
- **Model-based RL**: Learn a neural-network emulator of the QuTiP simulator for 100× faster rollouts.
- **Meta-RL**: Train across a distribution of qubit parameters so the agent adapts in a few shots to a new device.

### Hardware Integration
- Replace the QuTiP simulator with a **real QPU backend** (e.g., IBM Quantum via Qiskit Pulse) for hardware-in-the-loop RL.
- Use **randomized benchmarking** as the reward signal instead of analytical fidelity.

---

### Run Full Training

```bash
# Full training (~200k steps, ~15-30 min)
python train.py --timesteps 200000

# Monitor with TensorBoard
tensorboard --logdir logs/

# Evaluate & visualize results
python evaluate.py
```